# SAM-GA Optimization: Genetic Algorithm for Sensor Design

## Overview
This notebook demonstrates the use of Genetic Algorithm (GA) optimization for designing infrared microbolometer sensors. The goal is to find optimal basis function parameters (Gaussian means and standard deviations) that maximize the separability between different substances using Spectral Angle Mapper (SAM) distance metrics.

## Key Concepts
- **Basis Functions**: Gaussian curves defined by mean (µ) and standard deviation (σ) parameters
- **Chromosome Encoding**: Each chromosome represents 4 basis functions as [µ1, σ1, µ2, σ2, µ3, σ3, µ4, σ4]
- **Fitness Function**: Based on min-based dissimilarity score of the distance matrix between substances
- **Distance Metric**: Spectral Angle Mapper (SAM) for measuring spectral similarity
- **Optimization Target**: Maximize separability between white powder substances

## Objectives
1. Load spectral data for white powder substances
2. Set up GA parameters and fitness function
3. Run optimization to find optimal basis function parameters
4. Analyze results and visualize the best sensor designs
5. Perform diversity analysis of the final population

Notes:
- This notebook follows the standardized structure and documentation style
- Uses imports from the src package where possible
- Serves as a clean, professional reference for future experiment notebooks


## 1. Data Loading and Preprocessing

We begin by loading the spectral data for white powder substances and setting up the necessary parameters for sensor simulation and GA optimization.


In [ ]:
# Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pygad
from tqdm import tqdm
from pathlib import Path
import logging
import warnings

warnings.filterwarnings("ignore")

# Logging configuration
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Import from the package (only used symbols)
from lwi_microbolometer_design import (
    spectral_angle_mapper,
    generate_basis_from_chromosome,
    analyze_chromosome_diversity,
    vat_reorder,
    ivat_transform,
    visualize_distance_matrix_large,
    make_min_dissimilarity_fitness,
    generate_chromosome_distance_matrix,
    Chromosome,
)

# Plotting style
plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

logging.info("Imports and setup completed successfully")

In [ ]:
# Load substances and generate basis functions
# --------------------------------------------------------------------------------------------------

# Reproducibility
np.random.seed(42)

# File paths for input data
spectral_data_file = Path("../data/Test 3 - 4 White Powers/white_powders_with_labels.xlsx")
air_transmittance_file = Path("../data/Test 3 - 4 White Powers/Air transmittance.xlsx")

# Configurable Parameters
atmospheric_distance_ratio = (
    0.11  # Used to model the effect of atmospheric conditions on measurements
)
temperature_K = 293.15  # Temperature in Kelvin
air_refractive_index = 1  # Refractive index of air

# Load the spectral data of substances
substances_spectral_data = pd.read_excel(spectral_data_file)

# Extract the wavelength values (first column)
wavelengths = substances_spectral_data.iloc[:, :1].to_numpy()  # Shape = (d, 1)

# Extract substance labels (column names excluding the first)
substance_names = substances_spectral_data.columns[1:].to_numpy()

# Extract emissivity curves (all data except the first column)
emissivity_curves = substances_spectral_data.iloc[:, 1:].to_numpy()

# Load air transmittance matrix
air_transmittance = np.array(pd.read_excel(air_transmittance_file, header=None))
air_transmittance = air_transmittance[:, 1:]  # Remove the spectra column

logging.info(f"Loaded spectral data for {len(substance_names)} substances")
logging.info(f"Wavelength range: {wavelengths[0][0]:.1f} - {wavelengths[-1][0]:.1f} µm")
logging.info(f"Number of wavelength points: {len(wavelengths)}")

# Visualization: Spectral Emissivity Curves
plt.figure(figsize=(10, 6))
for name in substance_names:
    plt.plot(wavelengths, substances_spectral_data[name], label=name, linewidth=2)

plt.xlabel("Wavelength (µm)")
plt.ylabel("Emissivity Value")
plt.title("Spectral Emissivity Curves of White Powder Substances")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Genetic Algorithm Setup

We define the fitness function and GA parameters for optimizing the basis function parameters.


In [ ]:
fitness_func = make_min_dissimilarity_fitness(
    wavelengths=wavelengths,
    emissivity_curves=emissivity_curves,
    temperature_K=temperature_K,
    atmospheric_distance_ratio=atmospheric_distance_ratio,
    air_refractive_index=air_refractive_index,
    air_transmittance=air_transmittance,
    distance_metric=spectral_angle_mapper,
)
logging.info("fitness_func configured via factory")

## 3. GA Configuration and Execution

We configure the GA parameters and run the optimization to find optimal basis function parameters.


In [ ]:
# GA Configuration Parameters
# --------------------------------------------------------------------------------------------------

# Define parameter bounds
MU_MIN, MU_MAX = 4.0, 20.0
SIGMA_MIN, SIGMA_MAX = 0.1, 4.0

# Define the gene space for PyGAD's initialization
gene_space = []
for i in range(4):
    gene_space.append({"low": MU_MIN, "high": MU_MAX})  # Space for µ
    gene_space.append({"low": SIGMA_MIN, "high": SIGMA_MAX})  # Space for σ

# GA Parameters
POPULATION_SIZE = 200
NUM_GENERATIONS = 2000
NUM_PARENTS_MATING = 50  # Number of chromosomes to be selected as parents
NUM_GENES = 8  # 4 pairs of (µ, σ)

# GA Operator Settings
PARENT_SELECTION_TYPE = "sus"
CROSSOVER_TYPE = "uniform"
MUTATION_TYPE = "adaptive"
MUTATION_PROBABILITY = [0.5, 0.3]
KEEP_ELITISM = 5
RANDOM_SEED = 42

# Hall of Fame & Generational Tracking Setup
HALL_OF_FAME_SIZE = 100
hall_of_fame = []
generational_bests = []

# Lists to store fitness statistics per generation
best_fitness_per_gen = []
worst_fitness_per_gen = []
mean_fitness_per_gen = []

# Dictionary to store fitness scores at intervals
fitness_snapshots = {}

print("✅ GA configuration parameters set")

In [ ]:
# GA Callback Function
# --------------------------------------------------------------------------------------------------

# Create a progress bar instance (will be used by the callback)
pbar = tqdm(total=NUM_GENERATIONS, desc="Evolving Generations")


def on_generation(ga_instance: pygad.GA) -> None:
    """Per-generation GA callback.

    Args:
        ga_instance: Active PyGAD GA instance.
    """
    global \
        hall_of_fame, \
        generational_bests, \
        best_fitness_per_gen, \
        worst_fitness_per_gen, \
        mean_fitness_per_gen, \
        fitness_snapshots

    # Hall of Fame Logic
    population = ga_instance.population
    fitness = ga_instance.last_generation_fitness
    for i in range(len(population)):
        chromosome_tuple = (fitness[i], population[i].copy())
        if len(hall_of_fame) < HALL_OF_FAME_SIZE:
            hall_of_fame.append(chromosome_tuple)
            hall_of_fame.sort(key=lambda x: x[0])
        elif fitness[i] > hall_of_fame[0][0]:
            hall_of_fame[0] = chromosome_tuple
            hall_of_fame.sort(key=lambda x: x[0])

    # Generational Champions Logic
    gen_num = ga_instance.generations_completed
    if (gen_num % 20 == 0) and (0 < gen_num <= 200):
        chromosome, fitness, _ = ga_instance.best_solution()
        generational_bests.append((gen_num, fitness, chromosome.copy()))

    # Record Fitness Statistics
    last_gen_fitness = ga_instance.last_generation_fitness
    best_fitness_per_gen.append(np.max(last_gen_fitness))
    worst_fitness_per_gen.append(np.min(last_gen_fitness))
    mean_fitness_per_gen.append(np.mean(last_gen_fitness))

    # Take a snapshot of the full population's fitness every 500 generations
    if (gen_num % 500 == 0) and (gen_num > 0):
        fitness_snapshots[gen_num] = last_gen_fitness.copy()

    # Progress Bar Update Logic
    pbar.update(1)


logging.info("GA callback function defined")

In [ ]:
# Run the Genetic Algorithm
# --------------------------------------------------------------------------------------------------

logging.info("Running Genetic Algorithm... 🧬")

ga_instance = pygad.GA(
    num_generations=NUM_GENERATIONS,
    sol_per_pop=POPULATION_SIZE,
    num_parents_mating=NUM_PARENTS_MATING,
    num_genes=NUM_GENES,
    gene_space=gene_space,
    fitness_func=fitness_func,
    on_generation=on_generation,
    # Standard Recommended Settings
    parent_selection_type=PARENT_SELECTION_TYPE,
    crossover_type=CROSSOVER_TYPE,
    mutation_type=MUTATION_TYPE,
    mutation_probability=MUTATION_PROBABILITY,
    # Other Settings
    keep_elitism=KEEP_ELITISM,
    random_seed=RANDOM_SEED,
)

# Run the GA
ga_instance.run()

# Close the progress bar after the run is complete
pbar.close()
logging.info("GA run complete.")

## 4. Results Analysis and Visualization

We analyze the optimization results and visualize the best sensor designs found by the GA.


In [ ]:
# Best Solution Analysis
# --------------------------------------------------------------------------------------------------

# Retrieve the best solution
chromosome, solution_fitness, chromosome_idx = ga_instance.best_solution()

print("\n🏆 Best solution found (µ, σ pairs):")
for i in range(4):
    print(f"  Basis Function {i + 1}: µ = {chromosome[i * 2]:.3f}, σ = {chromosome[i * 2 + 1]:.3f}")
print(f"Fitness value of the best solution = {solution_fitness:.4f}")

# Plot Custom Fitness Statistics
plt.figure(figsize=(10, 6))
plt.plot(best_fitness_per_gen, label="Best Fitness", color="green")
plt.plot(mean_fitness_per_gen, label="Mean Fitness", color="blue")
plt.plot(worst_fitness_per_gen, label="Worst Fitness", color="red", linestyle="--")
plt.title("GA Fitness Statistics Over Generations", fontsize=16)
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Plot Best Basis Functions
best_basis_functions = generate_basis_from_chromosome(chromosome, wavelengths)

plt.figure(figsize=(10, 5))
for i in range(best_basis_functions.shape[1]):
    plt.plot(
        wavelengths,
        best_basis_functions[:, i],
        label=f"µ={chromosome[i * 2]:.2f}, σ={chromosome[i * 2 + 1]:.2f}",
    )

# Plot original substances in the background for context
for i, name in enumerate(substance_names):
    plt.plot(wavelengths, emissivity_curves[:, i], alpha=0.15)

plt.title("Best Basis Functions Found by GA", fontsize=16)
plt.xlabel("Wavelength (µm)")
plt.ylabel("Intensity / Emissivity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Diversity Analysis

We perform diversity analysis of the final population to understand the distribution of high-quality solutions.


In [ ]:
# Post-Run Diversity Analysis
# --------------------------------------------------------------------------------------------------


final_chromosomes = ga_instance.population
final_fitness = ga_instance.last_generation_fitness

final_population_objects = []
for genes, fitness in zip(final_chromosomes, final_fitness):
    final_population_objects.append(Chromosome(genes=genes, fitness=fitness))

F_threshold = 45.0  # Fitness threshold for high-quality chromosomes

analysis_report = analyze_chromosome_diversity(
    final_population=final_population_objects,
    top_n=100,
    f_threshold=F_threshold,
    r_peak=1,  # This radius might need tuning based on your distance values
)

# Print the human-readable summary
print("\n" + "=" * 40)
print("      Post-Run Diversity Analysis")
print("=" * 40)
print(f"Analysis of Top {analysis_report['total_chromosomes_analyzed']} Chromosomes:")
print(f"Found {analysis_report['high_quality_count']} chromosomes with score >= {F_threshold}.")
print(f"These chromosomes form approximately {analysis_report['family_count']} distinct families.")
print("-" * 40)

if not analysis_report["families"]:
    print("No distinct families found above the threshold.")
else:
    for family in analysis_report["families"]:
        print(f"Chromosome Family {family['family_id']}:")
        print(f"  - Top Score: {family['top_score']:.4f}")
        print(f"  - Members in Top 100: {family['member_count']}")

# Distance Matrix Visualization for High-Quality Chromosomes
if analysis_report["high_quality_chromosomes"]:
    high_quality_distance_matrix = generate_chromosome_distance_matrix(
        analysis_report["high_quality_chromosomes"]
    )

    # Compute the iVAT matrix
    vat_matrix, reorder = vat_reorder(high_quality_distance_matrix)
    ivat_matrix = ivat_transform(vat_matrix)

    # Visualize the original and iVAT matrices
    visualize_distance_matrix_large(
        high_quality_distance_matrix, title="Original Distance Matrix", figure_size=(4, 3)
    )
    visualize_distance_matrix_large(
        vat_matrix, title="VAT Matrix Visualization", figure_size=(4, 3)
    )
    visualize_distance_matrix_large(
        ivat_matrix, title="iVAT Matrix Visualization", figure_size=(4, 3)
    )
else:
    print("No high-quality chromosomes found for visualization.")

## Summary

This notebook demonstrates the complete workflow for optimizing infrared microbolometer sensor designs using Genetic Algorithms. Key achievements:

1. **Data Loading**: Successfully loaded spectral data for white powder substances
2. **GA Optimization**: Configured and ran GA with appropriate parameters
3. **Results Analysis**: Analyzed fitness distributions and identified best solutions
4. **Diversity Analysis**: Performed clustering analysis of high-quality chromosomes
5. **Visualization**: Created comprehensive visualizations of results

The optimized basis function parameters can be used to design sensors with improved separability between different substances.

### Functions Moved to src Package

The following general-purpose functions have been moved to the src package and are now imported:

- `generate_basis_from_chromosome` → `lwi_microbolometer_design.optimization`
- `distance_optimal_pairing` → `lwi_microbolometer_design.optimization`
- `generate_distance_matrix` → `lwi_microbolometer_design.optimization`
- `analyze_chromosome_diversity` → `lwi_microbolometer_design.optimization`
- `vat_reorder` → `lwi_microbolometer_design.analysis`
- `ivat_transform` → `lwi_microbolometer_design.analysis`
- `visualize_distance_matrix_large` → `lwi_microbolometer_design.visualization`

### Functions Remaining in Notebook

The following functions remain in the notebook as they are specific to this experiment:

- `fitness_func` - Requires access to local variables (wavelengths, emissivity_curves, etc.)
- `on_generation` - GA callback function specific to this experiment's configuration
- `Chromosome` class - Simple helper class for this experiment
